# FHOPS Onboarding — Orientation

This notebook orients a senior forest harvesting operations / planning user
to the **FHOPS** (Forest Harvesting Operations Planning System) codebase and
its public Python API and CLI.

## What FHOPS models

| Concept | Meaning |
| --- | --- |
| **Blocks** | Harvestable compartments with work-required, earliest/latest windows, and a landing ID. |
| **Machines** | Harvest systems: feller_buncher, grapple_skidder, roadside_processor, loader. |
| **Landings** | Forwarding endpoints with daily capacity limits. |
| **Harvest systems** | Job chains (felling -> transport -> processing -> loading) that bind machines to a role order. |
| **Productivity** | Per-machine-role, per-block productivity regressions (Lahrsen, Berry, TN-series). |
| **Mobilisation** | Machine relocation costs across blocks (walk threshold, flat move cost, distance matrix). |
| **Planning horizon** | A sequence of days, each split into shifts (S1, S2, …) with hours per shift. |

## This notebook's scope

- Locate the FHOPS repository from `Path.cwd()`.
- Discover bundled scenarios: `tiny7`, `small21`, `med42`, synthetic-small/medium/large.
- Load and inspect each scenario via `fhops.scenario.load_scenario` and `Problem.from_scenario`.
- Call the CLI `validate` command.
- Use the notebook-local AAM helpers (`diagnose_config`, `rtfm`, `explain_workflow`, `build_node`).

> **Note:** All AAM-style helpers in this notebook are *notebook-local* convenience
routines. They are **not** part of the public `fhops` package API.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "examples"))
from notebook_support import (
    find_repo_root,
    discover_scenarios,
    ScenarioKind,
    diagnose_config,
    inspect_scenario,
    rtfm,
    explain_workflow,
    build_node,
    run_fhops_cli,
)

repo_root = find_repo_root()
print(f'Repo root : {repo_root}')
print(f'CWD     : {Path.cwd()}')

Repo root : /srv/shared-data/gep/jupyterhub04-projects/fhops
CWD     : /srv/shared-data/gep/jupyterhub04-projects/fhops/examples


In [2]:
scenarios = discover_scenarios()
for kind, path in scenarios:
    print(f'{kind.value:20s}  {path}')

tiny7                 /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/tiny7/scenario.yaml
small21               /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/small21/scenario.yaml
med42                 /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/med42/scenario.yaml
synthetic-small       /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/synthetic/small/scenario.yaml
synthetic-medium      /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/synthetic/medium/scenario.yaml
synthetic-large       /srv/shared-data/gep/jupyterhub04-projects/fhops/examples/synthetic/large/scenario.yaml


## Inspect bundled scenarios via the Python API

We load each scenario through `fhops.scenario.io.load_scenario` and build
the runtime `Problem` via `Problem.from_scenario(scenario)`. The Problem
expands the horizon into concrete `(day, shift_id)` slots.

In [3]:
import pandas as pd

rows = []
for kind, path in scenarios:
    result = inspect_scenario(path)
    if not result['ok']:
        continue
    s = result['scenario']
    rows.append({
        'scenario': s['num_days'],
        'blocks': s['num_blocks'],
        'machines': s['num_machines'],
        'landings': s['num_landings'],
        'shifts': s['num_shifts'],
        'harvest_systems': ', '.join(s['harvest_systems'][:2]) if s['harvest_systems'] else '(default)',
    })

df = pd.DataFrame(rows, columns=['scenario', 'blocks', 'machines', 'landings', 'shifts', 'harvest_systems'])
df

,scenario,blocks,machines,landings,shifts,harvest_systems
0,7,2,9,2,7,ground_fb_skid
1,21,6,9,6,21,ground_fb_skid
2,42,12,9,12,42,ground_fb_skid
3,112,4,2,1,112,"cable_highlead_tn147, cable_micro_christie"
4,112,8,4,2,112,"cable_highlead_tn147, cable_micro_christie"
5,112,16,6,3,112,"cable_highlead_tn147, cable_micro_christie"


## Deep dive: tiny7 scenario

Tiny7 is a 7-day horizon with 2 blocks and 9 machines. It is the smallest
bundled scenario and ideal for quick iteration.

In [4]:
tiny7_path = Path(repo_root) / 'examples' / 'tiny7' / 'scenario.yaml'
tiny7 = inspect_scenario(tiny7_path)
pd.DataFrame(tiny7['scenario']['blocks_details'] if 'blocks_details' in tiny7['scenario'] else [],
             columns=['id', 'work_required', 'window'])

,id,work_required,window


## CLI validate

The CLI `validate` command runs the same loader and prints an entity summary.

In [5]:
for kind, path in scenarios[:3]:
    result = run_fhops_cli('validate', str(path))
    print(f'--- {kind.value} ---')
    print(result['stdout'][:300])
    print()

--- tiny7 ---
  Scenario: FHOPS   
       Tiny7        
┏━━━━━━━━━━┳━━━━━━━┓
┃ Entities ┃ Count ┃
┡━━━━━━━━━━╇━━━━━━━┩
│ Days     │ 7     │
│ Blocks   │ 2     │
│ Machines │ 9     │
│ Landings │ 2     │
└──────────┴───────┘


--- small21 ---
  Scenario: FHOPS   
      Small21       
┏━━━━━━━━━━┳━━━━━━━┓
┃ Entities ┃ Count ┃
┡━━━━━━━━━━╇━━━━━━━┩
│ Days     │ 21    │
│ Blocks   │ 6     │
│ Machines │ 9     │
│ Landings │ 6     │
└──────────┴───────┘


--- med42 ---
  Scenario: FHOPS   
      Medium42      
┏━━━━━━━━━━┳━━━━━━━┓
┃ Entities ┃ Count ┃
┡━━━━━━━━━━╇━━━━━━━┩
│ Days     │ 42    │
│ Blocks   │ 12    │
│ Machines │ 9     │
│ Landings │ 12    │
└──────────┴───────┘




## AAM helpers — review-only, no solver called

These return structured dictionaries with an explicit `executed=False`
flag for review-only callables and `executed=True` for ones that load real data.

In [6]:
diag = diagnose_config(tiny7_path)
print(f'ok={diag["ok"]}  name={diag["name"]}  status={diag["status"]}')
print(f'blocks={diag["scenario"]["num_blocks"]}  machines={diag["scenario"]["num_machines"]}  ' 
      f'shifts={diag["scenario"]["num_shifts"]}')

ok=True  name=FHOPS Tiny7  status=executed
blocks=2  machines=9  shifts=7


In [7]:
rtfm_result = rtfm('verify all onboarding symbols and doc paths')
print(f'Symbols verified: {rtfm_result["summary"]["symbols_verified"]}/{rtfm_result["summary"]["symbols_total"]}')
print(f'Docs verified   : {rtfm_result["summary"]["docs_verified"]}/{rtfm_result["summary"]["docs_total"]}')

Symbols verified: 13/13
Docs verified   : 6/6


In [8]:
workflow = explain_workflow('solve tiny7 and report KPIs')
print(workflow['status'], '| review_only =', workflow['review_only'])
for step in workflow['draft_steps']:
    print(f'  - {step}')

review_only | review_only = True
  - Load scenario via fhops.scenario.io.load_scenario
  - Build Problem via fhops.scenario.contract.Problem.from_scenario
  - Run fhops.optimization.heuristics.solve_sa (or CLI solve-heur)
  - Compute KPIs via fhops.evaluation.compute_kpis


In [9]:
node = build_node('assign all machines to block-1 on day 1')
print(f'executed={node["executed"]}  review_only={node["review_only"]}  ' 
      f'status={node["status"]}')

executed=False  review_only=True  status=review_only


## Next steps

- `01_fhops_operations_simulation.ipynb` — inspect and replay the operational simulation core.
- `02_fhops_solve_compare.ipynb` — solve tiny7/small21 with SA and compare objectives/KPIs.
- `03_fhops_playback_kpis.ipynb` — run playback and compute KPIs.
- `04_fhops_stochastic_what_if.ipynb` — stochastic playback with sampling.

All notebooks in this series are source-checkout-friendly and use modest
deterministic heuristic budgets (small `iters`, fixed `seed`). No licensed
solver is required.